In [1]:
import os
import re
import csv
import json

# ------------------------------------------------------------
# CONFIG
# ------------------------------------------------------------

PROPERTY_CSV = "complaints/properties.csv"
LOCATIONS_JS = "locations.js"

# ------------------------------------------------------------
# HELPERS
# ------------------------------------------------------------

def safe_slug(text):
    text = text.lower().strip()
    text = re.sub(r"[^\w\s-]", "", text)
    text = re.sub(r"\s+", "_", text)
    return text or "none"


def clean_field(val):
    """Remove control characters from CSV fields."""
    if val is None:
        return ""

    cleaned = re.sub(r"[\x00-\x08\x0b\x0c\x0e-\x1f\x7f-\x9f]", "", str(val))
    return cleaned.strip()


# ------------------------------------------------------------
# LOAD locations.js
# ------------------------------------------------------------

def load_js_locations(path):
    """
    Reads:

    const stvrLocations = [
        {...},
        {...}
    ];

    and returns (locations, original_text)
    """

    if not os.path.exists(path):
        return [], "const stvrLocations = [];"

    with open(path, "r", encoding="utf-8") as f:
        content = f.read()

    match = re.search(
        r"const\s+stvrLocations\s*=\s*(\[[\s\S]*?\])\s*;",
        content,
        re.DOTALL,
    )

    if not match:
        raise ValueError("Could not locate stvrLocations array.")

    js_array = match.group(1)

    # --------------------------------------------------------
    # Remove ONLY block comments.
    # Do NOT remove // because it destroys https:// URLs.
    # --------------------------------------------------------

    js_array = re.sub(
        r"/\*[\s\S]*?\*/",
        "",
        js_array,
        flags=re.DOTALL,
    )

    # Remove trailing commas
    js_array = re.sub(r",(\s*[}\]])", r"\1", js_array)

    try:
        locations = json.loads(js_array)

    except json.JSONDecodeError as e:

        print("\n================ JSON ERROR ================")
        print(e)

        start = max(0, e.pos - 150)
        end = min(len(js_array), e.pos + 150)

        print("\nContext around error:\n")
        print(repr(js_array[start:end]))
        print("\n============================================\n")

        raise

    return locations, content


# ------------------------------------------------------------
# SAVE locations.js
# ------------------------------------------------------------

def save_js_locations(path, original_js, locations):

    new_array = json.dumps(
        locations,
        indent=2,
        ensure_ascii=False
    )

    pattern = r"const\s+stvrLocations\s*=\s*\[[\s\S]*?\]\s*;"

    replacement = f"const stvrLocations = {new_array};"

    if re.search(pattern, original_js, re.DOTALL):
        new_js = re.sub(
            pattern,
            replacement,
            original_js,
            flags=re.DOTALL,
        )
    else:
        new_js = replacement

    with open(path, "w", encoding="utf-8", newline="\n") as f:
        f.write(new_js)


# ------------------------------------------------------------
# LOAD EXISTING locations.js
# ------------------------------------------------------------

try:
    js_locations, original_js = load_js_locations(LOCATIONS_JS)

except Exception as e:
    print(f"\n✖ Could not load locations.js")
    print(e)
    print("\nCreating a fresh locations.js...\n")

    js_locations = []
    original_js = "const stvrLocations = [];"

# ------------------------------------------------------------
# INDEX EXISTING LOCATIONS
# ------------------------------------------------------------

js_by_address = {}

for loc in js_locations:
    addr = str(loc.get("address", "")).strip().lower()
    if addr:
        js_by_address[addr] = loc

# ------------------------------------------------------------
# LOAD PROPERTY CSV
# ------------------------------------------------------------

if not os.path.exists(PROPERTY_CSV):
    print(f"✖ Error: {PROPERTY_CSV} not found.")
    raise SystemExit(1)

with open(PROPERTY_CSV, "r", encoding="utf-8-sig", newline="") as f:

    reader = csv.DictReader(f)

    for row in reader:

        address = clean_field(row.get("Address", ""))

        if not address:
            continue

        key = address.lower()

        lat = clean_field(row.get("Latitude", ""))
        lng = clean_field(row.get("Longitude", ""))

        airbnb = clean_field(row.get("Airbnb URL", ""))
        vrbo = clean_field(row.get("VRBO URL", ""))
        other = clean_field(row.get("Other Listing Site", ""))

        # complaints_url = (
        #     f"data/complaints/{safe_slug(address)}/index.html"
        # )


        addr_slug = safe_slug(address)

        complaints_url = ""

        address_folder = os.path.join("complaints", addr_slug)

        if os.path.isdir(address_folder):

            # Look for the first unit folder containing an index.html
            for item in sorted(os.listdir(address_folder)):
                unit_path = os.path.join(address_folder, item)

                if (
                    os.path.isdir(unit_path)
                    and os.path.exists(os.path.join(unit_path, "index.html"))
                ):
                    complaints_url = (
                        f"data/complaints/{addr_slug}/{item}/index.html"
                    )
                    break

        # Fallback if no complaint page exists yet
        if not complaints_url:
            complaints_url = f"data/complaints/{addr_slug}/"



        if key in js_by_address:

            loc = js_by_address[key]

            if airbnb and not loc.get("airbnbUrl"):
                loc["airbnbUrl"] = airbnb

            if vrbo and not loc.get("vrboUrl"):
                loc["vrboUrl"] = vrbo

            if other and not loc.get("otherUrl"):
                loc["otherUrl"] = other

            loc["viewComplaintsUrl"] = complaints_url

            if lat:
                try:
                    loc["lat"] = float(lat)
                except ValueError:
                    pass

            if lng:
                try:
                    loc["lng"] = float(lng)
                except ValueError:
                    pass

        else:

            new_loc = {
                "title": "Licensed STVR Property",
                "address": address,
                "license": clean_field(row.get("Permit Number", "")),
                "lat": float(lat) if lat else None,
                "lng": float(lng) if lng else None,
                "vrboUrl": vrbo,
                "airbnbUrl": airbnb,
                "otherUrl": other,
                "viewComplaintsUrl": complaints_url,
            }

            js_locations.append(new_loc)
            js_by_address[key] = new_loc

# ------------------------------------------------------------
# SORT BY ADDRESS
# ------------------------------------------------------------

js_locations.sort(
    key=lambda x: x.get("address", "").lower()
)

# ------------------------------------------------------------
# SAVE
# ------------------------------------------------------------

save_js_locations(
    LOCATIONS_JS,
    original_js,
    js_locations,
)

print(f"\n✓ Successfully wrote {len(js_locations)} properties to {LOCATIONS_JS}")


✓ Successfully wrote 2 properties to locations.js


In [1]:
import os
import re
import csv
import json

# ------------------------------------------------------------
# CONFIG
# ------------------------------------------------------------

PROPERTY_CSV = os.path.join("complaints", "properties.csv")
LOCATIONS_JS = "locations.js"

# ------------------------------------------------------------
# HELPERS
# ------------------------------------------------------------

def safe_slug(text):
    text = text.lower().strip()
    text = re.sub(r"[^\w\s-]", "", text)
    text = re.sub(r"\s+", "_", text)
    return text or "none"


def clean_field(val):
    """Remove control characters from CSV fields."""
    if val is None:
        return ""

    cleaned = re.sub(r"[\x00-\x08\x0b\x0c\x0e-\x1f\x7f-\x9f]", "", str(val))
    return cleaned.strip()


# ------------------------------------------------------------
# LOAD locations.js
# ------------------------------------------------------------

def load_js_locations(path):
    """
    Reads:

    const stvrLocations = [
        {...},
        {...}
    ];

    and returns (locations, original_text)
    """

    if not os.path.exists(path):
        return [], "const stvrLocations = [];"

    with open(path, "r", encoding="utf-8") as f:
        content = f.read()

    match = re.search(
        r"const\s+stvrLocations\s*=\s*(\[[\s\S]*?\])\s*;",
        content,
        re.DOTALL,
    )

    if not match:
        raise ValueError("Could not locate stvrLocations array.")

    js_array = match.group(1)

    # --------------------------------------------------------
    # Remove ONLY block comments.
    # Do NOT remove // because it destroys https:// URLs.
    # --------------------------------------------------------

    js_array = re.sub(
        r"/\*[\s\S]*?\*/",
        "",
        js_array,
        flags=re.DOTALL,
    )

    # Remove trailing commas
    js_array = re.sub(r",(\s*[}\]])", r"\1", js_array)

    try:
        locations = json.loads(js_array)

    except json.JSONDecodeError as e:

        print("\n================ JSON ERROR ================")
        print(e)

        start = max(0, e.pos - 150)
        end = min(len(js_array), e.pos + 150)

        print("\nContext around error:\n")
        print(repr(js_array[start:end]))
        print("\n============================================\n")

        raise

    return locations, content


# ------------------------------------------------------------
# SAVE locations.js
# ------------------------------------------------------------

def save_js_locations(path, original_js, locations):

    new_array = json.dumps(
        locations,
        indent=2,
        ensure_ascii=False
    )

    pattern = r"const\s+stvrLocations\s*=\s*\[[\s\S]*?\]\s*;"

    replacement = f"const stvrLocations = {new_array};"

    if re.search(pattern, original_js, re.DOTALL):
        new_js = re.sub(
            pattern,
            replacement,
            original_js,
            flags=re.DOTALL,
        )
    else:
        new_js = replacement

    with open(path, "w", encoding="utf-8", newline="\n") as f:
        f.write(new_js)


# ------------------------------------------------------------
# LOAD EXISTING locations.js
# ------------------------------------------------------------

try:
    js_locations, original_js = load_js_locations(LOCATIONS_JS)

except Exception as e:
    print(f"\n✖ Could not load locations.js")
    print(e)
    print("\nCreating a fresh locations.js...\n")

    js_locations = []
    original_js = "const stvrLocations = [];"

# ------------------------------------------------------------
# INDEX EXISTING LOCATIONS
# ------------------------------------------------------------

js_by_address = {}

for loc in js_locations:
    addr = str(loc.get("address", "")).strip().lower()
    if addr:
        js_by_address[addr] = loc

# ------------------------------------------------------------
# LOAD PROPERTY CSV
# ------------------------------------------------------------

if not os.path.exists(PROPERTY_CSV):
    print(f"✖ Error: {PROPERTY_CSV} not found.")
    raise SystemExit(1)

with open(PROPERTY_CSV, "r", encoding="utf-8-sig", newline="") as f:

    reader = csv.DictReader(f)

    for row in reader:

        address = clean_field(row.get("Address", ""))

        if not address:
            continue

        key = address.lower()

        lat = clean_field(row.get("Latitude", ""))
        lng = clean_field(row.get("Longitude", ""))

        airbnb = clean_field(row.get("Airbnb URL", ""))
        vrbo = clean_field(row.get("VRBO URL", ""))
        other = clean_field(row.get("Other Listing Site", ""))

        addr_slug = safe_slug(address)
        complaints_url = ""
        address_folder = os.path.join("complaints", addr_slug)

        if os.path.isdir(address_folder):
            # 1. Check if index.html exists directly at the property folder level
            direct_index = os.path.join(address_folder, "index.html")
            if os.path.exists(direct_index):
                complaints_url = f"data/complaints/{addr_slug}/index.html"
            else:
                # 2. Search subdirectories (units or 'none') for an index.html
                subdirs = sorted([d for d in os.listdir(address_folder) if os.path.isdir(os.path.join(address_folder, d))])
                
                found_unit = None
                for item in subdirs:
                    unit_path = os.path.join(address_folder, item)
                    if os.path.exists(os.path.join(unit_path, "index.html")):
                        if item.lower() != "none":
                            found_unit = item
                            break
                        elif not found_unit:
                            found_unit = item

                if found_unit:
                    if found_unit.lower() == "none":
                        complaints_url = f"data/complaints/{addr_slug}/none/index.html"
                    else:
                        complaints_url = f"data/complaints/{addr_slug}/{found_unit}/index.html"

        # 3. Fallback if no complaint page exists yet
        if not complaints_url:
            complaints_url = f"data/complaints/{addr_slug}/"


        if key in js_by_address:

            loc = js_by_address[key]

            if airbnb and not loc.get("airbnbUrl"):
                loc["airbnbUrl"] = airbnb

            if vrbo and not loc.get("vrboUrl"):
                loc["vrboUrl"] = vrbo

            if other and not loc.get("otherUrl"):
                loc["otherUrl"] = other

            loc["viewComplaintsUrl"] = complaints_url

            if lat:
                try:
                    loc["lat"] = float(lat)
                except ValueError:
                    pass

            if lng:
                try:
                    loc["lng"] = float(lng)
                except ValueError:
                    pass

        else:

            new_loc = {
                "title": "Licensed STVR Property",
                "address": address,
                "license": clean_field(row.get("Permit Number", "")),
                "lat": float(lat) if lat else None,
                "lng": float(lng) if lng else None,
                "vrboUrl": vrbo,
                "airbnbUrl": airbnb,
                "otherUrl": other,
                "viewComplaintsUrl": complaints_url,
            }

            js_locations.append(new_loc)
            js_by_address[key] = new_loc

# ------------------------------------------------------------
# SORT BY ADDRESS
# ------------------------------------------------------------

js_locations.sort(
    key=lambda x: x.get("address", "").lower()
)

# ------------------------------------------------------------
# SAVE
# ------------------------------------------------------------

save_js_locations(
    LOCATIONS_JS,
    original_js,
    js_locations,
)

print(f"\n✓ Successfully wrote {len(js_locations)} properties to {LOCATIONS_JS}")


✓ Successfully wrote 2 properties to locations.js
